# Step 2b: praeclarum + fixes (keep the XXXX)

Tests the two fixes on the model we chose:
1. **Spelling fix**: praeclarum's CDLI spelling (`_kug-ud_`, `sz`, `s,`) converted so CuneiScribe recognises every sign.
2. **Redactions kept**: each CFPB redaction (`XXXX`, `XX/XX/XXXX`, `{$XXXX}`) becomes one broken sign `x`, the scholarly mark for an illegible sign, instead of sending the model into a loop.

Two ways of keeping the XXXX are compared:
- **split**: translate the text around the redaction, put the broken sign back in place
- **single**: replace each XXXX with a lone `x` and translate the whole line

`Runtime → Run all`.

In [ ]:
!pip -q install cuneiscribe sentencepiece protobuf
!apt-get -qq install -y fonts-noto-core > /dev/null

In [ ]:
%%writefile nanni.py
import re

# CFPB redactions: XXXX, XX/XX/XXXX, {$XXXX}, XXXX-XXXX ...
REDACTION = re.compile(r"\{?\$?X{2,}(?:[/\-.]X{2,})*\}?")
GAP = "x"            # ATF convention: an illegible / broken sign
GAP_SIGN = "[x]"     # how a gap shows up in the cuneiform string (rendered later)

def normalize_atf(a):
    """praeclarum (CDLI ATF) spelling -> CuneiScribe spelling."""
    a = re.sub(r"_([^_]+)_", lambda m: m.group(1).upper(), a)   # _kug-ud_ -> KUG-UD (logograms)
    a = a.replace("_", "")
    for k, v in [("sz", "š"), ("SZ", "Š"), ("s,", "ṣ"), ("S,", "Ṣ"), ("t,", "ṭ"), ("T,", "Ṭ")]:
        a = a.replace(k, v)
    return a

def translate_with_redactions(line, to_akkadian, mode="split"):
    """Keep every redaction as one broken sign.
    mode="split": translate the text between redactions, rejoin with a gap.
    mode="single": collapse each redaction to a lone 'x' and translate the whole line."""
    if mode == "single":
        return normalize_atf(to_akkadian(REDACTION.sub("x", line)))
    parts = REDACTION.split(line)
    out = []
    for i, part in enumerate(parts):
        if part.strip(" ,.;:"):
            out.append(normalize_atf(to_akkadian(part.strip())))
        if i < len(parts) - 1:
            out.append(GAP)
    return " ".join(out)

def to_cuneiform(akk, conv):
    words = [GAP_SIGN if w == GAP else conv.to_cuneiform(w) for w in akk.split()]
    return " ".join(words)

def coverage(cuneiform):
    toks = re.findall(r"\[[^\]]+\]|[\U00012000-\U0001254F]", cuneiform)
    toks = [t for t in toks if t != GAP_SIGN]            # redactions aren't failures
    if not toks:
        return 0.0, []
    bad = [t for t in toks if t.startswith("[")]
    return round(100 * (1 - len(bad) / len(toks)), 1), bad

def runaway(akk, source):
    """True if the model looped: very long output or mostly one repeated word."""
    w = akk.split()
    return len(akk) > 4 * max(len(source), 20) or (len(w) > 6 and max(w.count(x) for x in set(w)) / len(w) > 0.5)

In [ ]:
TEST_LINES = [
    "I have called them XXXX times and nobody will help me.",
    "They charged me an overdraft fee even though I had money in my account.",
    "Why am I being treated with such contempt after paying on time for years?",
    "On XX/XX/XXXX I paid {$XXXX} and XXXX bank still says I owe them.",
]
MODEL = "praeclarum/cuneiform"

In [ ]:
# STEP 2b TEST
from cuneiscribe import AkkadianTranslator, CuneiformConverter, validate
from nanni import *
tr, conv = AkkadianTranslator(MODEL), CuneiformConverter()
ak = lambda s: tr.to_akkadian(s, max_length=128)   # cap length so a loop can't run away

rows = []
for line in TEST_LINES:
    for mode in ["split", "single"]:
        a = translate_with_redactions(line, ak, mode=mode)
        c = to_cuneiform(a, conv)
        cov, bad = coverage(c)
        rows.append(dict(english=line, mode=mode, akkadian=a,
                         back_translation=tr.to_english(a, max_length=128),
                         cuneiform=c, gaps=c.count(GAP_SIGN), sign_coverage_pct=cov,
                         unmapped=" ".join(bad), runaway=runaway(a, line),
                         validator=validate(a, line).suggestion))
        print(f"[{mode}] {line}\n  AK  : {a}\n  CU  : {c}\n  BACK: {rows[-1]['back_translation']}\n")

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame(rows)
df.to_csv("step2b_results.csv", index=False)
df[["english","mode","akkadian","back_translation","gaps","sign_coverage_pct","runaway"]]

## Pass criteria
1. **No runaway loops** (`runaway` = False on every row).
2. **Gaps match redactions**: line 1 has 1 gap, line 4 has 3, lines 2–3 have 0.
3. **Sign coverage ≥ 90%** (gaps don't count against it).
4. **Lines 2–3 still read as grievances** in the back-translation.

Then pick **split** or **single** for the redacted lines: whichever keeps more of the grievance.

In [ ]:
# Preview tablets for the redacted lines (both modes). Gaps currently show as empty boxes:
# that's a placeholder, the real look of a redaction gets designed at the ASCII stage.
from cuneiscribe import TabletRenderer
from IPython.display import Image, display
r = TabletRenderer()
for i, row in df.iterrows():
    fn = f"step2b_{i+1}_{row['mode']}.png"
    r.render_png(row["cuneiform"], title=f"[{row['mode']}] {row['english'][:70]}", output_path=fn)
    display(Image(fn))

In [ ]:
!zip -q step2b_outputs.zip step2b_results.csv step2b_*.png
from google.colab import files
files.download("step2b_outputs.zip")